<a href="https://colab.research.google.com/github/EmirhanByr/stock-market-elt-pipeline/blob/main/01_extract_stock_datas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance

import yfinance as yf
import pandas as pd
from datetime import datetime

# 1. EXTRACT: Borsa İstanbul (BIST) lokomotif hisselerinin son 1 aylık canlı verisini çekelim
hisseler = ["THYAO.IS", "ASELS.IS", "TUPRS.IS", "GARAN.IS", "EREGL.IS"]
veri_listesi = []

for sembol in hisseler:
    # Yahoo Finance API'den son 1 aylık (1mo) günlük veriyi çekiyoruz
    gecmis = yf.Ticker(sembol).history(period="1mo").reset_index()
    gecmis["hisse_kodu"] = sembol.replace(".IS", "") # '.IS' uzantısını temizleyelim
    veri_listesi.append(gecmis)

# Tüm hisseleri tek bir tabloda birleştirelim
ham_df = pd.concat(veri_listesi, ignore_index=True)

# 2. TRANSFORM (Ön İşleme ve Mühendislik Standartları):
# Sadece ihtiyacımız olan sütunları seçip standart 'snake_case' isimlendirmeye çevirelim
df = pd.DataFrame()
df["islem_tarihi"] = pd.to_datetime(ham_df["Date"]).dt.strftime("%Y-%m-%d")
df["hisse_kodu"] = ham_df["hisse_kodu"]
df["acilis_fiyati"] = ham_df["Open"].round(2)
df["en_yuksek"] = ham_df["High"].round(2)
df["en_dusuk"] = ham_df["Low"].round(2)
df["kapanis_fiyati"] = ham_df["Close"].round(2)
df["islem_hacmi"] = ham_df["Volume"].astype("int64")

# Finansal Metrik: Gün içi değişim yüzdesi ((Kapanış - Açılış) / Açılış * 100)
df["gunluk_degisim_yuzdesi"] = (((df["kapanis_fiyati"] - df["acilis_fiyati"]) / df["acilis_fiyati"]) * 100).round(2)

# Veri soyu (Data Lineage) için boru hattının çalışma zamanı
df["yuklenme_tarihi"] = datetime.now().strftime("%Y-%m-%d")

# 3. LOAD HAZIRLIĞI: İndeks sütunu olmadan (index=False) Parquet olarak kaydetme
dosya_adi = "borsa_gunluk_veriler.parquet"
df.to_parquet(dosya_adi, index=False)

print(f"✅ '{dosya_adi}' başarıyla oluşturuldu! Toplam satır sayısı: {len(df)}")
df.head(10)

✅ 'borsa_gunluk_veriler.parquet' başarıyla oluşturuldu! Toplam satır sayısı: 115


,islem_tarihi,hisse_kodu,acilis_fiyati,en_yuksek,en_dusuk,kapanis_fiyati,islem_hacmi,gunluk_degisim_yuzdesi,yuklenme_tarihi
0,2026-09-16,THYAO,287.25,288.00,258.25,273.00,79653407,-4.96,2026-10-19
1,2026-09-17,THYAO,276.50,290.00,273.50,289.25,72724185,4.61,2026-10-19
2,2026-09-18,THYAO,289.00,291.00,283.25,285.50,35977826,-1.21,2026-10-19
3,2026-09-21,THYAO,287.50,294.00,282.50,293.50,51991406,2.09,2026-10-19
4,2026-09-22,THYAO,293.50,302.75,291.25,298.00,54419270,1.53,2026-10-19
5,2026-09-23,THYAO,298.75,302.25,297.25,298.50,34210549,-0.08,2026-10-19
6,2026-09-24,THYAO,298.25,300.50,288.50,288.50,26198900,-3.27,2026-10-19
7,2026-09-25,THYAO,292.25,294.00,289.50,290.75,20893753,-0.51,2026-10-19
8,2026-09-28,THYAO,288.00,293.00,286.00,287.00,28590319,-0.35,2026-10-19
9,2026-09-29,THYAO,288.00,294.25,287.50,291.50,27797456,1.22,2026-10-19
